<a href="https://colab.research.google.com/github/IvanLemos5000/clearbank-analise/blob/main/desafio_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Desafio Final — Análise Financeira ClearBank

**Autor:** (seu nome)

**Objetivo:** ler `transacoes.csv`, validar registros, gerar métricas mensais, sinalizar transações suspeitas e exportar `relatorio.json`.

## Estrutura do notebook
1. Imports e constantes
2. `ler_transacoes()` — leitura via `csv.DictReader`
3. `validar_transacao()` — valida uma linha
4. `processar_transacoes()` — separa válidas/inválidas
5. `gerar_relatorio()` — métricas por mês
6. `identificar_suspeitas()` e `calcular_periodo()`
7. `salvar_json()` — exporta
8. `exibir_relatorio()` — formata terminal
9. Célula principal — orquestra tudo

## Regras de validação
- `id`: inteiro não vazio
- `cliente_id`: não vazio
- `data`: AAAA-MM-DD
- `tipo`: `credito` ou `debito`
- `valor`: float > 0

> Antes de executar, garanta que o arquivo `transacoes.csv` esteja na mesma pasta do notebook.

In [7]:
import csv
import json
from datetime import datetime

# ---------- Constantes ----------
LIMITE_SUSPEITO = 10000.00
ARQUIVO_CSV = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"

print("Configuração carregada. Limite de suspeita:", LIMITE_SUSPEITO)

Configuração carregada. Limite de suspeita: 10000.0


## 1) Leitura do arquivo CSV (módulo nativo `csv`)

Usa `csv.DictReader` para acessar as colunas pelo nome. Trata `FileNotFoundError` e `OSError`.

In [8]:
def ler_transacoes(caminho):
    """Lê o CSV e retorna uma lista de dicionários brutos.
    Retorna lista vazia se o arquivo não existir."""
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            return list(reader)
    except FileNotFoundError:
        print(f"[ERRO] Arquivo não encontrado: {caminho}")
        return []
    except OSError as e:
        print(f"[ERRO] Falha ao abrir arquivo: {e}")
        return []


# --- Teste rápido ---
linhas = ler_transacoes(ARQUIVO_CSV)
print(f"Linhas brutas lidas: {len(linhas)}")
print("Primeira linha:", linhas[0] if linhas else "—")

Linhas brutas lidas: 21
Primeira linha: {'id': '1', 'data': '2026-01-05', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500.00', 'descricao': 'Salario janeiro', 'categoria': 'salario'}


## 2) Validação e limpeza dos dados

Valida cada linha individualmente. Erros em uma linha são capturados e a linha é descartada — o processamento continua.

Contém **3 blocos `try/except`** distintos: `int(id)`, `datetime.strptime(data)` e `float(valor)`.

In [9]:
def validar_transacao(linha):
    """Valida uma única linha. Retorna dict limpo ou None se inválida."""

    # --- id ---
    id_txt = (linha.get("id") or "").strip()
    if not id_txt:
        return None
    try:
        id_int = int(id_txt)
    except ValueError:
        return None

    # --- cliente_id ---
    cliente_id = (linha.get("cliente_id") or "").strip()
    if not cliente_id:
        return None

    # --- data (try/except #1) ---
    data_txt = (linha.get("data") or "").strip()
    try:
        data_obj = datetime.strptime(data_txt, "%Y-%m-%d")
    except ValueError:
        return None

    # --- tipo ---
    tipo = (linha.get("tipo") or "").strip().lower()
    if tipo not in ("credito", "debito"):
        return None

    # --- valor (try/except #2) ---
    valor_txt = (linha.get("valor") or "").strip()
    try:
        valor = float(valor_txt)
    except ValueError:
        return None
    if valor <= 0:
        return None

    return {
        "id": id_int,
        "data": data_obj,
        "mes": data_obj.strftime("%Y-%m"),
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": (linha.get("descricao") or "").strip(),
        "categoria": (linha.get("categoria") or "").strip(),
    }


# --- Testes rápidos ---
print(validar_transacao({"id":"1","data":"2026-01-05","cliente_id":"CLI001",
                         "tipo":"credito","valor":"3500.00",
                         "descricao":"x","categoria":"y"}))
print(validar_transacao({"id":"abc","data":"2026-01-05","cliente_id":"CLI001",
                         "tipo":"credito","valor":"10","descricao":"","categoria":""}))
print(validar_transacao({"id":"3","data":"xx-xx","cliente_id":"CLI001",
                         "tipo":"credito","valor":"10","descricao":"","categoria":""}))

{'id': 1, 'data': datetime.datetime(2026, 1, 5, 0, 0), 'mes': '2026-01', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': 3500.0, 'descricao': 'x', 'categoria': 'y'}
None
None


In [10]:
def processar_transacoes(linhas_brutas):
    """Separa em válidas (lista) e inválidas (contagem)."""
    validas, invalidas = [], 0
    for linha in linhas_brutas:
        registro = validar_transacao(linha)
        if registro is None:
            invalidas += 1
        else:
            validas.append(registro)
    return validas, invalidas


# --- Teste rápido ---
validas_teste, invalidas_teste = processar_transacoes(ler_transacoes(ARQUIVO_CSV))
print(f"Válidas: {len(validas_teste)} | Inválidas: {invalidas_teste}")

Válidas: 14 | Inválidas: 7


## 3) Agrupamento mensal e métricas

Agrupa por mês (`AAAA-MM`) e calcula: quantidade, total crédito, total débito, saldo, média, maior e menor valor.

In [11]:
def gerar_relatorio(transacoes):
    """Agrupa por mês (AAAA-MM) e retorna dict de métricas."""
    resumo = {}
    for t in transacoes:
        mes = t["mes"]
        if mes not in resumo:
            resumo[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "soma_valores": 0.0,
                "maior": t,
                "menor": t,
            }
        r = resumo[mes]
        r["quantidade"] += 1
        r["soma_valores"] += t["valor"]
        if t["tipo"] == "credito":
            r["total_credito"] += t["valor"]
        else:
            r["total_debito"] += t["valor"]
        if t["valor"] > r["maior"]["valor"]:
            r["maior"] = t
        if t["valor"] < r["menor"]["valor"]:
            r["menor"] = t

    final = {}
    for mes, r in sorted(resumo.items()):
        final[mes] = {
            "quantidade": r["quantidade"],
            "total_credito": round(r["total_credito"], 2),
            "total_debito": round(r["total_debito"], 2),
            "saldo": round(r["total_credito"] - r["total_debito"], 2),
            "media": round(r["soma_valores"] / r["quantidade"], 2),
            "maior_valor": round(r["maior"]["valor"], 2),
            "menor_valor": round(r["menor"]["valor"], 2),
        }
    return final


# --- Teste rápido ---
resumo_teste = gerar_relatorio(validas_teste)
for mes, dados in resumo_teste.items():
    print(mes, dados)

2026-01 {'quantidade': 4, 'total_credito': 7700.0, 'total_debito': 430.5, 'saldo': 7269.5, 'media': 2032.62, 'maior_valor': 4200.0, 'menor_valor': 180.5}
2026-02 {'quantidade': 4, 'total_credito': 18800.0, 'total_debito': 1520.0, 'saldo': 17280.0, 'media': 5080.0, 'maior_valor': 15000.0, 'menor_valor': 320.0}
2026-03 {'quantidade': 6, 'total_credito': 20000.0, 'total_debito': 850.65, 'saldo': 19149.35, 'media': 3475.11, 'maior_valor': 12000.0, 'menor_valor': 99.9}


## 4) Transações suspeitas e período analisado

Marca como suspeita toda transação com `valor > LIMITE_SUSPEITO`. Também calcula o intervalo de dias entre a transação mais antiga e a mais recente.

In [12]:
def identificar_suspeitas(transacoes):
    """Retorna transações cujo valor ultrapassa LIMITE_SUSPEITO."""
    return [t for t in transacoes if t["valor"] > LIMITE_SUSPEITO]


def calcular_periodo(transacoes):
    """Retorna (data_min, data_max, dias_entre_elas)."""
    datas = [t["data"] for t in transacoes]
    data_min, data_max = min(datas), max(datas)
    return data_min, data_max, (data_max - data_min).days


# --- Teste rápido ---
suspeitas_teste = identificar_suspeitas(validas_teste)
print("Suspeitas:", [(t["id"], t["valor"]) for t in suspeitas_teste])
print("Período:", calcular_periodo(validas_teste))

Suspeitas: [(5, 15000.0), (15, 12000.0)]
Período: (datetime.datetime(2026, 1, 5, 0, 0), datetime.datetime(2026, 3, 25, 0, 0), 79)


## 5) Exportação para JSON

Salva o relatório em `relatorio.json` com `ensure_ascii=False` (acentos preservados) e indentação de 2 espaços.

In [13]:
def salvar_json(caminho, dados):
    """Salva o relatório em JSON com acentos preservados."""
    try:
        with open(caminho, "w", encoding="utf-8") as f:
            json.dump(dados, f, ensure_ascii=False, indent=2, default=str)
        print(f"[OK] Relatório salvo em {caminho}")
    except OSError as e:
        print(f"[ERRO] Não foi possível salvar JSON: {e}")

## 6) Exibição formatada no terminal

Formata valores no padrão brasileiro (`R$ 3.500,00`) e organiza o relatório com separadores visuais.

In [14]:
def formatar_moeda(valor):
    """Formata 3500.5 -> 'R$ 3.500,50' (padrão brasileiro)."""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(resumo, suspeitas, validas, invalidas, data_min, data_max):
    sep = "=" * 35
    print(sep)
    print("RELATÓRIO MENSAL")
    print(sep)
    print(f"Período analisado: {data_min.strftime('%Y-%m-%d')} → {data_max.strftime('%Y-%m-%d')}")
    print(f"Transações válidas:   {len(validas)}")
    print(f"Transações inválidas: {invalidas}")
    print()
    for mes, r in resumo.items():
        print(f"Mês: {mes}")
        print(f"  Transações:    {r['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(r['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(r['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(r['saldo'])}")
        print(f"  Média:         {formatar_moeda(r['media'])}")
        print(f"  Maior valor:   {formatar_moeda(r['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(r['menor_valor'])}")
        print()
    print(sep)
    print("TRANSAÇÕES SUSPEITAS")
    print(sep)
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for t in suspeitas:
            print(f"ID: {t['id']} | Cliente: {t['cliente_id']} | "
                  f"Data: {t['data'].strftime('%Y-%m-%d')} | Valor: {formatar_moeda(t['valor'])}")

## 7) Célula de Execução Principal

Orquestra todas as funções: leitura → validação → métricas → JSON → relatório formatado.

In [15]:
def main():
    print("Lendo arquivo...")
    linhas_brutas = ler_transacoes(ARQUIVO_CSV)

    total_lidas = len(linhas_brutas)
    validas, invalidas = processar_transacoes(linhas_brutas)

    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Linhas válidas:        {len(validas)}")
    print(f"Linhas inválidas:      {invalidas}")
    print()

    if not validas:
        print("Nenhuma transação válida para processar.")
        return

    data_min, data_max, dias = calcular_periodo(validas)
    print(f"Dias entre a transação mais antiga e a mais recente: {dias}\n")

    resumo = gerar_relatorio(validas)
    suspeitas = identificar_suspeitas(validas)

    exibir_relatorio(resumo, suspeitas, validas, invalidas, data_min, data_max)

    dados_json = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": len(validas),
        "total_transacoes_invalidas": invalidas,
        "periodo": {
            "inicio": data_min.strftime("%Y-%m-%d"),
            "fim": data_max.strftime("%Y-%m-%d"),
            "dias": dias,
        },
        "resumo_mensal": resumo,
        "transacoes_suspeitas": [
            {
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime("%Y-%m-%d"),
                "valor": t["valor"],
            }
            for t in suspeitas
        ],
    }
    salvar_json(ARQUIVO_JSON, dados_json)


if __name__ == "__main__":
    main()

Lendo arquivo...
Total de linhas lidas: 21
Linhas válidas:        14
Linhas inválidas:      7

Dias entre a transação mais antiga e a mais recente: 79

RELATÓRIO MENSAL
Período analisado: 2026-01-05 → 2026-03-25
Transações válidas:   14
Transações inválidas: 7

Mês: 2026-01
  Transações:    4
  Total crédito: R$ 7.700,00
  Total débito:  R$ 430,50
  Saldo:         R$ 7.269,50
  Média:         R$ 2.032,62
  Maior valor:   R$ 4.200,00
  Menor valor:   R$ 180,50

Mês: 2026-02
  Transações:    4
  Total crédito: R$ 18.800,00
  Total débito:  R$ 1.520,00
  Saldo:         R$ 17.280,00
  Média:         R$ 5.080,00
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 320,00

Mês: 2026-03
  Transações:    6
  Total crédito: R$ 20.000,00
  Total débito:  R$ 850,65
  Saldo:         R$ 19.149,35
  Média:         R$ 3.475,11
  Maior valor:   R$ 12.000,00
  Menor valor:   R$ 99,90

TRANSAÇÕES SUSPEITAS
ID: 5 | Cliente: CLI003 | Data: 2026-02-14 | Valor: R$ 15.000,00
ID: 15 | Cliente: CLI005 | Data: 202